# 03 — Estabelecer referências mínimas em validation

## Objetivo
Responder se features simples de cor aprendem sinal além da classe majoritária, antes de qualquer CNN.

## Entradas
Certificado do NB02 e somente train/validation v2.

## Saídas
`reports/experimental_v2/baselines/<experiment_id>/`: design, ambiente, proveniência, métricas por classe, predições e confusion matrices.

## Permissões de dados
Somente train para fit de features aprendidas/scaler/modelo; validation para métricas. Nenhuma leitura de calibration, holdout ou final test.

## Reprodutibilidade
Seed 42; features RGB 128×128, 16 bins/canal e momentos; scaler ajustado somente em train. Registro de critérios desativado até revisão explícita.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.v2_data import load_development
from scripts.v2_baselines import run_classical_study, FEATURE_CONFIG, LOGREG_CONFIG
from scripts.v2_artifacts import verify_bundle
EXECUTE_BASELINES = True
EXISTING_BASELINE_DIRECTORY = None  # caminho relativo explícito para reler um resultado
EXPERIMENT_ID = run_id("baselines_v2")
REGISTER_CRITERIA = False
frames, certificate = load_development(ROOT)
display(pd.Series({k: len(v) for k, v in frames.items()}).to_frame("imagens"))
display({"features": FEATURE_CONFIG, "logreg": LOGREG_CONFIG, "split_version": certificate["split_version"]})


## Hipótese e comparação pré-definida
**Majority:** memorizar a classe mais frequente de train mede o quanto accuracy pode parecer alta apenas pelo desbalanceamento.

**Regressão logística:** histogramas RGB e média/desvio por canal descrevem cor, sem aprender representações profundas.
`StandardScaler` e regressão logística com pesos de classe usam somente train. A métrica primária é macro-F1 com sete classes fixas.
Accuracy é contextual; o relatório por classe mostra o custo do balanceamento. Não há busca de hiperparâmetros neste notebook.


In [ ]:
baseline = None
baseline_dir = None
if EXECUTE_BASELINES:
    baseline_dir = ROOT / "reports/experimental_v2/baselines" / EXPERIMENT_ID
    baseline = run_classical_study(ROOT, baseline_dir, EXPERIMENT_ID)
elif EXISTING_BASELINE_DIRECTORY:
    from scripts.v2_artifacts import relative_path
    baseline_dir = relative_path(ROOT, EXISTING_BASELINE_DIRECTORY)
    verify_bundle(baseline_dir)
    baseline = read_json(baseline_dir / "baseline_results.json")
else:
    print("Execução desativada; nenhum resultado presumido.")


## Resultados e interpretação
O intervalo por grupo não trata vistas da mesma lesão como independentes. É um intervalo de validation, condicionado ao processo de desenvolvimento.
Falha de convergência fica registrada e impede usar a referência para registrar critérios.


In [ ]:
if baseline is not None:
    table = pd.DataFrame([{ "baseline": name, "macro_f1": r["validation"]["macro_f1"],
                            "accuracy": r["validation"]["accuracy"], "weighted_f1": r["validation"]["weighted_f1"],
                            "fit_seconds": r["fit_seconds"], "converged": r.get("converged", True)}
                          for name, r in baseline["results"].items()])
    display(table)
    from scripts.v2_metrics import plot_confusion
    for name, result in baseline["results"].items():
        display(pd.DataFrame(result["validation"]["per_class"]).T)
        plot_confusion(result["validation"], f"Validation v2 — {name}"); plt.show()
    display(Markdown(f"Delta macro-F1 LogReg − majority: **{baseline['macro_f1_delta']:.4f}**. "
                     "Esta comparação informa uma referência de desenvolvimento, não desempenho clínico."))


## Critérios do próximo experimento: decisão explícita
O artefato de baseline pode fundamentar o gate v2. A proposta simples é exigir média de macro-F1 entre seeds ao menos igual à melhor referência acima.
`REGISTER_CRITERIA=False` mantém essa decisão pendente; não transporta o antigo limite 0,60.
Ao revisar e ativar a célula, o critério recebe hash e data **antes** do registro do plano CNN. Nenhum piso clínico de recall é inventado.


In [ ]:
criteria_path = None
if REGISTER_CRITERIA:
    if baseline_dir is None:
        raise ValueError("Execute ou selecione explicitamente um baseline v2 antes de registrar critérios")
    from scripts.v2_training import register_baseline_criteria
    criteria_path = ROOT / "reports/experimental_v2/criteria" / f"{EXPERIMENT_ID}.json"
    display(register_baseline_criteria(ROOT, baseline_dir, criteria_path))
else:
    print("Critério operacional pendente de revisão; métricas de baseline não promovem modelos automaticamente.")
if baseline_dir is not None:
    print("Artefatos:", baseline_dir.relative_to(ROOT).as_posix())


## Conclusão
A conclusão empírica é apresentada acima quando há execução. Uma accuracy alta de majority é esperada em dados desbalanceados; macro-F1 e recall por classe são necessários para interpretá-la.

## Artefatos produzidos
Design e resultados em diretório novo, com hashes, seed, ambiente e identidade do split. Modelos lineares são exportados como arrays/JSON, sem pickle.

## Próxima etapa
NB04: uma ablação controlada; após revisão dos critérios, NB05 registra o plano de CNN.
